In [6]:
!pip install -U ultralytics

!pip install roboflow
!pip install --upgrade pip


In [17]:
from roboflow import Roboflow
rf = Roboflow(api_key="e2GxpareEObD10qdBv76")
project = rf.workspace("ejakunskas-gmail-com").project("oat-detection")
version = project.version(11)
dataset = version.download("yolov11")
                

loading Roboflow workspace...
loading Roboflow project...


In [19]:
import torch
version.deploy("yolov11", "runs/classify/train32")

View the status of your deployment at: https://app.roboflow.com/ejakunskas-gmail-com/oat-detection/11
Share your model with the world at: https://universe.roboflow.com/ejakunskas-gmail-com/oat-detection/model/11


In [8]:
import os
import shutil
import random

# Define the base paths for the dataset directories
BASE_DIR = '/media/edward/HDD/Docker-Workspace/YOLOv8/oat-detection-11'

# Directories to process
DATA_SPLITS = ['train', 'valid', 'test']

# Class mapping for directories
CLASS_MAPPING = {
    '0': 'Leaf',
    '1': 'Noise',
    '2': 'Panicle',
    '3': 'PanicleClump'
}

# Train/valid/test split ratios
SPLIT_RATIOS = {'train': 0.7, 'valid': 0.2, 'test': 0.1}

def create_directories():
    """
    Create directories for classification datasets if they do not exist.
    """
    for split in DATA_SPLITS:
        for class_name in CLASS_MAPPING.values():
            dir_path = os.path.join(BASE_DIR, split, class_name)
            os.makedirs(dir_path, exist_ok=True)

def process_label_file(label_path):
    """
    Process a single label file to extract the class ID.
    
    Args:
        label_path (str): Path to the label file.
    
    Returns:
        str: The class name corresponding to the first class ID in the label file.
    """
    try:
        with open(label_path, 'r') as file:
            line = file.readline().strip()
            class_id = line.split(' ')[0]  # Get the first number which is the class ID
            if class_id in CLASS_MAPPING:
                return CLASS_MAPPING[class_id]
    except Exception as e:
        print(f"Error processing label file {label_path}: {e}")
    return None

def copy_image_to_class_directory(image_path, split, class_name):
    """
    Copy the image to the correct split directory.
    
    Args:
        image_path (str): Path to the image file.
        split (str): The current dataset split (train, valid, test).
        class_name (str): The class name to place the image under.
    """
    dest_dir = os.path.join(BASE_DIR, split, class_name)
    dest_path = os.path.join(dest_dir, os.path.basename(image_path))
    
    # Copy the image to the new location
    try:
        shutil.copy2(image_path, dest_path)
        print(f"Copied {image_path} to {dest_path}")
    except Exception as e:
        print(f"Error copying {image_path} to {dest_path}: {e}")

def process_dataset():
    """
    Process the entire dataset to convert it from segmentation to classification.
    """
    images_by_class = {class_name: [] for class_name in CLASS_MAPPING.values()}

    label_dir = os.path.join(BASE_DIR, 'train', 'labels')
    if not os.path.exists(label_dir):
        print(f"Label directory does not exist: {label_dir}")
        return
    
    label_files = [os.path.join(label_dir, f) for f in os.listdir(label_dir) if f.endswith('.txt')]
    print(f"Processing {len(label_files)} label files in {label_dir}")
    
    for label_path in label_files:
        class_name = process_label_file(label_path)
        if class_name is None:
            print(f"Skipping label file {label_path} due to missing class mapping.")
            continue
        
        image_name = os.path.splitext(os.path.basename(label_path))[0] + '.jpg'
        image_path = os.path.join(BASE_DIR, 'train', 'images', image_name)
        
        if not os.path.isfile(image_path):
            print(f"Image file not found for label file {label_path}. Expected at {image_path}")
            continue

        images_by_class[class_name].append(image_path)

    # Shuffle and split images into train, valid, and test sets
    for class_name, images in images_by_class.items():
        random.shuffle(images)
        total_images = len(images)
        
        train_size = int(total_images * SPLIT_RATIOS['train'])
        valid_size = int(total_images * SPLIT_RATIOS['valid'])
        
        train_images = images[:train_size]
        valid_images = images[train_size:train_size + valid_size]
        test_images = images[train_size + valid_size:]

        print(f"Class {class_name}: Total = {total_images}, Train = {len(train_images)}, Valid = {len(valid_images)}, Test = {len(test_images)}")
        
        for image_path in train_images:
            copy_image_to_class_directory(image_path, 'train', class_name)
        for image_path in valid_images:
            copy_image_to_class_directory(image_path, 'valid', class_name)
        for image_path in test_images:
            copy_image_to_class_directory(image_path, 'test', class_name)

def delete_original_folders():
    """
    Delete the original images and labels folders.
    """
    image_dir = os.path.join(BASE_DIR, 'train', 'images')
    label_dir = os.path.join(BASE_DIR, 'train', 'labels')
    try:
        if os.path.exists(image_dir):
            shutil.rmtree(image_dir)
            print(f"Deleted image directory: {image_dir}")
        if os.path.exists(label_dir):
            shutil.rmtree(label_dir)
            print(f"Deleted label directory: {label_dir}")
    except Exception as e:
        print(f"Error deleting directories: {e}")

def main():
    """
    Main function to run the dataset conversion process.
    """
    random.seed(42)  # Ensure reproducibility
    create_directories()
    process_dataset()
    delete_original_folders()

if __name__ == "__main__":
    main()


Label directory does not exist: /media/edward/HDD/Docker-Workspace/YOLOv8/oat-detection-11/train/labels


In [9]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.1


In [15]:

from ultralytics import YOLO
import os
print(os.getcwd())


# Load a model
#model = YOLO("yolov8n.yaml")  # build a new model from YAML
model = YOLO("yolo11x-cls.pt")  # load a pretrained model (recommended for training)
#model = YOLO("yolov8n.yaml").load("yolov8n.pt")  # build from YAML and transfer weights

# Train the model
results = model.train(data="/media/edward/HDD/Docker-Workspace/YOLOv8/oat-detection-11", epochs=30, batch=12, workers=10, imgsz=640, patience=0)

/media/edward/HDD/Docker-Workspace/YOLOv8
Ultralytics 8.3.74 🚀 Python-3.11.11 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 24102MiB)
engine/trainer: task=classify, mode=train, model=yolo11x-cls.pt, data=/media/edward/HDD/Docker-Workspace/YOLOv8/oat-detection-11, epochs=30, time=None, patience=0, batch=12, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=10, project=None, name=train32, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False

train: Scanning /media/edward/HDD/Docker-Workspace/YOLOv8/oat-detection-11/train... 1682 images, 0 corrupt: 100%|██████████| 1682/1682 [00:00<?, ?it/s]
val: Scanning /media/edward/HDD/Docker-Workspace/YOLOv8/oat-detection-11/test... 245 images, 0 corrupt: 100%|██████████| 245/245 [00:00<?, ?it/s]


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.00125, momentum=0.9) with parameter groups 82 weight(decay=0.0), 83 weight(decay=0.00046875), 83 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 10 dataloader workers
Logging results to runs/classify/train32
Starting training for 30 epochs...

      Epoch    GPU_mem       loss  Instances       Size


               classes   top1_acc   top5_acc: 100%|██████████| 11/11 [00:00<00:00, 15.51it/s]

                   all      0.841          1


...

      30/30      9.32G     0.1983          2        640: 100%|██████████| 141/141 [00:14<00:00,  9.49it/s]
               classes   top1_acc   top5_acc: 100%|██████████| 11/11 [00:00<00:00, 15.72it/s]

                   all      0.943          1



30 epochs completed in 0.143 hours.
Optimizer stripped from runs/classify/train32/weights/last.pt, 57.0MB
Optimizer stripped from runs/classify/train32/weights/best.pt, 57.0MB

Validating runs/classify/train32/weights/best.pt...
Ultralytics 8.3.74 🚀 Python-3.11.11 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 24102MiB)
YOLO11x-cls summary (fused): 227 layers, 28,337,540 parameters, 0 gradients, 110.3 GFLOPs
WARNING ⚠️ Dataset 'split=val' not found, using 'split=test' instead.
train: /media/edward/HDD/Docker-Workspace/YOLOv8/oat-detection-11/train... found 1682 images in 4 classes ✅ 
val: None...
test: /media/edward/HDD/Docker-Workspace/YOLOv8/oat-detection-11/test... found 245 images in 4 classes ✅ 


               classes   top1_acc   top5_acc: 100%|██████████| 11/11 [00:00<00:00, 15.58it/s]


                   all      0.959          1
Speed: 0.4ms preprocess, 2.3ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/train32


In [11]:
version.deploy("yolov11", "training1")

In [12]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Function to pad the image with half-patch size
def pad_image(image, patch_height, patch_width):
    pad_top = patch_height // 2
    pad_left = patch_width // 2
    padded_image = cv2.copyMakeBorder(image, pad_top, pad_top, pad_left, pad_left, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return padded_image

# Function to split the image into patches with dynamic sizes
def split_image(image, rows, cols):
    img_height, img_width, _ = image.shape
    patch_height = img_height // rows
    patch_width = img_width // cols

    patches = []
    for r in range(rows):
        for c in range(cols):
            patch = image[r*patch_height:(r+1)*patch_height, c*patch_width:(c+1)*patch_width]
            patches.append(patch)
    
    return patches

# Function to process patches and display both the mask overlay and object removal on the original image
def process_patches_and_display(patch_results, original_patches, rows, cols, patch_size):
    """
    Process patches by applying the mask overlay and removing detected objects. Then display both.
    """
    patch_height, patch_width = patch_size
    patches_with_overlay = []
    patches_without_objects = []
    stitched_mask = np.zeros((patch_height * rows, patch_width * cols), dtype=np.uint8)  # Initialize binary mask for the whole image

    for i, (result, original_patch) in enumerate(zip(patch_results, original_patches)):
        patch_mask = np.zeros((patch_height, patch_width, 3), dtype=np.uint8)  # Empty mask for each patch
        binary_mask = np.zeros((patch_height, patch_width), dtype=np.uint8)  # Binary mask for object removal

        if result[0].masks is not None:
            masks = result[0].masks.data.cpu().numpy()  # Get YOLOv8 masks

            for mask in masks:
                # Resize the mask to match the patch size
                mask_resized = cv2.resize(mask, (patch_width, patch_height))

                mask_binary = (mask_resized * 255).astype(np.uint8)  # Convert to binary mask
                mask_colored = np.zeros_like(patch_mask)
                mask_colored[mask_binary > 0] = (255, 0, 0)  # Color detected mask areas red

                # Add the colored mask to the patch mask
                patch_mask = cv2.addWeighted(patch_mask, 1, mask_colored, 0.5, 0)

                # Apply morphological transformation to smooth mask artifacts
                kernel = np.ones((5, 5), np.uint8)
                mask_binary = cv2.morphologyEx(mask_binary, cv2.MORPH_CLOSE, kernel)

                # Mark the outer 1-pixel border of the patch in the binary mask
                binary_mask[0, :] = 255  # Top edge
                binary_mask[-1, :] = 255  # Bottom edge
                binary_mask[:, 0] = 255  # Left edge
                binary_mask[:, -1] = 255  # Right edge

                # Remove the detected objects without resizing the mask
                binary_mask = cv2.bitwise_or(binary_mask, mask_binary)

        # Blend the original image with the mask (overlay)
        patch_with_overlay = cv2.addWeighted(original_patch, 0.6, patch_mask, 0.4, 0)
        patches_with_overlay.append(patch_with_overlay)

        # Remove objects by replacing them with black (where binary mask > 0)
        binary_mask_3channel = cv2.cvtColor(binary_mask, cv2.COLOR_GRAY2BGR)
        patch_without_objects = np.where(binary_mask_3channel > 0, (0, 0, 0), original_patch)
        patches_without_objects.append(patch_without_objects)

        # Stitch the binary mask into the whole mask
        row_start = (i // cols) * patch_height
        col_start = (i % cols) * patch_width
        stitched_mask[row_start:row_start + patch_height, col_start:col_start + patch_width] = cv2.bitwise_or(
            stitched_mask[row_start:row_start + patch_height, col_start:col_start + patch_width], binary_mask
        )

    return patches_without_objects, patches_with_overlay, stitched_mask

# Perform segmentation with given rows, cols, and stagger
def perform_segmentation(input_image, trained_model, rows, cols, stagger=False):
    img_height, img_width, _ = input_image.shape
    patch_size = (img_height // rows, img_width // cols)

    if stagger:
        # Pad the image with half-patch size
        padded_image = pad_image(input_image, patch_size[0], patch_size[1])
        patches = split_image(padded_image, rows + 1, cols + 1)  # +1 to account for padding
    else:
        patches = split_image(input_image, rows=rows, cols=cols)

    patch_results = []

    for patch in patches:
        patch_pil = Image.fromarray(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
        results = trained_model.predict(source=patch_pil)
        patch_results.append(results)

    patches_without_objects, patches_with_overlay, stitched_mask = process_patches_and_display(
        patch_results, patches, rows + 1 if stagger else rows, cols + 1 if stagger else cols, patch_size
    )
    
    return stitched_mask

# Function to remove padding from the mask
def remove_padding(mask, patch_height, patch_width):
    pad_top = patch_height // 2
    pad_left = patch_width // 2
    return mask[pad_top:-pad_top, pad_left:-pad_left]  # Remove the padding from all sides

# Load YOLOv8 model
input_image = cv2.imread('/workspace/Data/masked_rgb_cam18443010715BEE0F00_29_08_2024_11_06_22_X360.1_Y296.0_Z690.1_RX180.0_RY0.0_RZ180.0_55.jpg')
trained_model = YOLO('/workspace/YOLOv8/runs/classify/train143/weights/best.pt')

# First segmentation with rows=3, cols=6
mask_1 = perform_segmentation(input_image, trained_model, rows=3, cols=6, stagger=False)

# Second segmentation with rows=3, cols=6, but with staggering by padding the image
mask_2 = perform_segmentation(input_image, trained_model, rows=3, cols=6, stagger=True)
mask_2 = remove_padding(mask_2, patch_size[0], patch_size[1])

# Combine both masks using logical OR operation to create a more robust final mask
combined_mask = cv2.bitwise_or(mask_1, mask_2)

# Visualization of masks overlayed on the original image
def visualize_overlay(image, mask_1, mask_2, combined_mask):
    # Convert masks to 3-channel for overlaying on original image
    mask_1_3channel = cv2.cvtColor(mask_1, cv2.COLOR_GRAY2BGR)
    mask_2_3channel = cv2.cvtColor(mask_2, cv2.COLOR_GRAY2BGR)
    combined_mask_3channel = cv2.cvtColor(combined_mask, cv2.COLOR_GRAY2BGR)

    # Overlay mask_1 on original image
    overlay_mask_1 = cv2.addWeighted(image, 0.7, mask_1_3channel, 0.3, 0)

    # Overlay mask_2 on original image
    overlay_mask_2 = cv2.addWeighted(image, 0.7, mask_2_3channel, 0.3, 0)

    # Overlay combined_mask on original image
    overlay_combined_mask = cv2.addWeighted(image, 0.7, combined_mask_3channel, 0.3, 0)

    # Plot the images
    plt.figure(figsize=(15,10))

    # Show mask_1 overlay
    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(overlay_mask_1, cv2.COLOR_BGR2RGB))
    plt.title('Mask 1 Overlay')
    cv2.imwrite('overlay_mask_1.jpg', overlay_mask_1)

    # Show mask_2 overlay
    plt.subplot(1, 3, 2)
    plt.imshow(cv2.cvtColor(overlay_mask_2, cv2.COLOR_BGR2RGB))
    plt.title('Mask 2 Overlay')
    cv2.imwrite('overlay_mask_2.jpg', overlay_mask_2)

    # Show combined mask overlay
    plt.subplot(1, 3, 3)
    plt.imshow(cv2.cvtColor(overlay_combined_mask, cv2.COLOR_BGR2RGB))
    plt.title('Combined Mask Overlay')

    plt.show()

# Visualize the masks overlayed on the original image
visualize_overlay(input_image, mask_1, mask_2, combined_mask)

# Save final images
cv2.imwrite('combined_mask.jpg', combined_mask)
cv2.imwrite('final_image_no_objects.jpg', final_image_no_objects)


[ WARN:0@940.339] global loadsave.cpp:241 findDecoder imread_('/workspace/Data/masked_rgb_cam18443010715BEE0F00_29_08_2024_11_06_22_X360.1_Y296.0_Z690.1_RX180.0_RY0.0_RZ180.0_55.jpg'): can't open/read file: check file path/integrity


FileNotFoundError: [Errno 2] No such file or directory: '/workspace/YOLOv8/runs/classify/train143/weights/best.pt'

In [ ]:
import cv2
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# Function to calculate the RGB metric and ignore black pixels
def calculate_rgb_metric(image):
    # Split the image into its red, green, and blue components
    blue, green, red = cv2.split(image.astype(float) + 1e-6)  # Add small value to avoid division by zero
    
    # Calculate the metric: (green - red) / (red + green)
    metric = (green - red) / (red + green)
    
    # Mask out black pixels (where all channels are 0)
    mask_non_black = (red > 10) | (green > 10) | (blue > 10)
    metric[~mask_non_black] = np.nan  # Set black pixels to NaN
    
    return metric, mask_non_black

# Function to compute statistics for non-black pixels
def compute_statistics(metric):
    # Remove NaN values (black pixels)
    valid_pixels = metric[~np.isnan(metric)]
    
    # Compute statistics
    mode_val = stats.mode(valid_pixels) if len(valid_pixels) > 0 else None
    mean_val = np.mean(valid_pixels) if len(valid_pixels) > 0 else None
    median_val = np.median(valid_pixels) if len(valid_pixels) > 0 else None
    max_val = np.max(valid_pixels) if len(valid_pixels) > 0 else None
    min_val = np.min(valid_pixels) if len(valid_pixels) > 0 else None
    
    return mode_val, mean_val, median_val, max_val, min_val

# Load the image
image = cv2.imread('final_image_no_objects.jpg')

# Calculate the RGB metric and get mask of non-black pixels
metric, mask_non_black = calculate_rgb_metric(image)

# Compute statistics
mode_val, mean_val, median_val, max_val, min_val = compute_statistics(metric)

# Print statistics
print(f'Mode: {mode_val}')
print(f'Mean: {mean_val}')
print(f'Median: {median_val}')
print(f'Max: {max_val}')
print(f'Min: {min_val}')

# Plot the metric ignoring NaN values using a masked array
masked_metric = np.ma.masked_where(np.isnan(metric), metric)

# Set up normalization to scale metric values between -1 and 1 for plotting


# Create a colormap where NaN values are transparent
cmap = plt.cm.hot
cmap.set_bad(color='black')  # Set the color for NaN values

# Plot the image
plt.imshow(masked_metric, cmap=cmap)
plt.title('RGB Metric Visualization (Excluding Black Pixels)')
plt.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=plt.gca())
plt.show()

# Optionally save the image
plt.imsave('output_image.png', masked_metric, cmap=cmap)
